# TalentDesk, Section 3 Lab (Exercise): Coordinator, Subagents, and Adaptive Decomposition

A hands-on exercise built on the **Claude Agent SDK**, running **Sonnet**
(`claude-sonnet-4-6`). It combines the two Section 3 skills: a **hub-and-spoke** design where
one **coordinator** decomposes a request and delegates each slice to a focused **subagent**
(Lab 1), and **parallel dispatch** with **adaptive decomposition** that you **measure** against
a fixed pipeline (Lab 2). You fill in four short `TODO` blocks; everything else is provided. An
offline mock coordinator lets you watch delegation and parallel dispatch without a key or
Node.js, and a full solution is at the end.

## The real-world scenario

TalentDesk's single recruiter agent has grown into a tangle: it knows the screening bar, the
interview-scheduling rules, and how to calm an upset candidate, all in one prompt, and the rules
bleed into each other. The fix is a **coordinator** that owns the big picture and hands each
concern to a focused **subagent**: one for screening, one for scheduling, one for candidate care.
The catch: a subagent starts with **no memory of the coordinator**, so you must pass it everything
it needs, on purpose.

And at end of day there is a **batch** of candidates to triage. Handling them one after another is
slow, and forcing every request through the same fixed steps does needless work (screening a
candidate nobody asked to screen). Two ideas fix this: **fan out** independent work so subagents
run at once, and **decompose adaptively** so each request gets only the steps it needs.

The question this lab answers: **how do you split a mixed recruiter request across focused
subagents, give each only the context it needs, run independent work in parallel, and prove that
adaptive routing beats a fixed pipeline?**

## Objectives

- Build a **coordinator** plus three **subagents** (screening, scheduling, care) with
  `AgentDefinition`, delegating through the **Agent tool** (`allowed_tools` must include
  `"Agent"`).
- Decompose by **query type**: `classify` a request into intents, then `route` only to the
  subagents it needs, instead of a fixed pipeline.
- Pass **structured context** (content plus metadata) explicitly, since a subagent inherits
  nothing.
- Dispatch subagents **in parallel** for a batch, and **evaluate** adaptive decomposition against
  a fixed pipeline with a small rubric.

## The outcome you should reach

By the end you will have:

- a `classify` and `route` that send a screening-only request to one subagent and a mixed request
  to exactly the subagents it needs;
- a self-contained hand-off packet (content plus metadata) that gives a subagent the candidate
  facts it cannot otherwise see;
- an offline coordinator that narrates delegation and flags **parallel dispatch** on a batch;
- and a scorer showing adaptive decomposition beating the fixed pipeline on a labelled set.

Target time: **20 to 30 minutes.** Four small `TODO` blocks, all pure Python and testable
offline. The live coordinator needs a real key and Node.js 18+.

## How to run

Run top to bottom. The classifier, router, hand-off, and evaluation are pure Python and run
anywhere; an offline **mock coordinator** narrates delegation and parallel dispatch using your own
router. To run the real coordinator, paste a real key into **Setup 2/3**, re-run from the top, and
have Node.js 18+ installed. Model behaviour is not perfectly deterministic, so treat each live run
as an observation.

## 0. Setup

**This cell:** installs the packages. The **Agent SDK** runs the coordinator and subagents;
the base SDK and dotenv handle the key. The Agent SDK also needs Node.js 18+, which cannot be
pip-installed; the offline mock does not need it.

In [ ]:
# ===== SETUP 1/3 - install the Agent SDK =====
%pip install -q claude-agent-sdk anthropic python-dotenv

**This cell:** imports, the model, the `RUN_LIVE` switch, and `run_async()` so the async
coordinator can be called like a normal function.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, and an async runner =====
import os                                       # read the API key from the environment
import sys                                      # detect Windows (it needs a special event loop)
import re                                       # pull candidate ids out of requests
import json                                     # build and print the context payloads
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv             #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the coordinator will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}
    def worker():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:    box["value"] = loop.run_until_complete(make_coro())
        except Exception as e: box["error"] = e
        finally: loop.close()
    t = threading.Thread(target=worker); t.start(); t.join()
    if "error" in box: raise box["error"]
    return box.get("value")

print("live model calls:", "ON" if RUN_LIVE else "OFF (mock coordinator narrates delegation)")

**This cell:** the shared **TalentDesk world**: a small set of candidates with the facts a
subagent might need (stage, whether they cleared screening, years, skills). The coordinator reasons
over this, and it is the source of the metadata we pass in a hand-off.

In [ ]:
# ===== SETUP 3/3 - the shared candidate data =====
CANDIDATES = {                                   # the batch to triage
    "C1": {"stage": 3, "cleared": True,  "years": 5, "skills": ["python", "backend"]},
    "C3": {"stage": 1, "cleared": True,  "years": 4, "skills": ["python", "backend"]},
    "C4": {"stage": 2, "cleared": True,  "years": 6, "skills": ["python", "backend", "aws"]},
}
STAGE_NAMES = {1: "applied", 2: "screening", 3: "interview", 4: "offer"}   # code -> human word
print("candidates:", list(CANDIDATES))

### The hub-and-spoke pattern

The **coordinator** is the hub: it sees the whole request, decides how to split it, delegates each
slice, and merges the answers. Each **subagent** is a spoke: its own prompt, its own tools, its own
fresh context. Two rules make it work: the **Agent tool does the delegation**, so include `"Agent"`
in `allowed_tools`; and **nothing is shared unless you pass it**, since a subagent starts with zero
coordinator memory.

**This cell:** binds the Agent SDK pieces (provided). Live, it imports them. Offline, it
defines light stand-ins so the coordinator objects below still construct and the notebook stays
runnable without the package.

In [ ]:
# ===== bind the Agent SDK pieces: real when available, stand-ins otherwise (provided) =====
try:
    from claude_agent_sdk import (query, ClaudeAgentOptions, AgentDefinition,
                                  AssistantMessage, ResultMessage, TextBlock, ToolUseBlock)
    SDK_OK = True
except Exception:
    SDK_OK = False
    class AgentDefinition:                        # stores the spoke spec (description/prompt/tools)
        def __init__(self, description="", prompt="", tools=None, model=None):
            self.description, self.prompt = description, prompt
            self.tools, self.model = tools or [], model
    class ClaudeAgentOptions:                     # stores the coordinator settings
        def __init__(self, model=None, system_prompt=None, agents=None, allowed_tools=None, **kw):
            self.model, self.system_prompt = model, system_prompt
            self.agents, self.allowed_tools = agents or {}, allowed_tools or []
print("Agent SDK importable:", SDK_OK)

**This cell:** the three **subagents** as `AgentDefinition`s (provided). Each has a narrow
`description` (which the coordinator reads to route), a narrow `prompt` (its role), a tool list, and
a model. Keeping the concerns separate is what stops their rules from mixing.

In [ ]:
# ===== three spokes: screening, scheduling, care (provided) =====
screening_agent = AgentDefinition(
    description="Evaluates a candidate against the job bar. Use for screening and resume review.",
    prompt="You screen ONE candidate against the bar. Never schedule or console. Be brief.",
    tools=["Read"], model="sonnet")
scheduling_agent = AgentDefinition(
    description="Arranges interviews and slots. Use for scheduling and availability requests.",
    prompt="You handle scheduling only. Never screen or console. Be brief.",
    tools=["Read"], model="sonnet")
care_agent = AgentDefinition(
    description="Handles upset or ghosted candidates. Use when the tone is frustrated or urgent.",
    prompt="You de-escalate calmly, apologise once, and flag a human. Be brief.",
    tools=["Read"], model="sonnet")
print("defined:", ["screening_agent", "scheduling_agent", "care_agent"])

**This cell:** the **coordinator** (the hub), provided. It registers all three spokes under
`agents`, has a system prompt that says to decompose, delegate, and combine, and puts `"Agent"` in
`allowed_tools` so it may actually spawn them.

In [ ]:
# ===== the coordinator (hub), provided =====
COORD = ClaudeAgentOptions(
    model=MODEL,
    system_prompt=("You are the TalentDesk coordinator. Split each request into parts, delegate "
                   "screening to screening-agent, scheduling to scheduling-agent, and frustrated "
                   "or ghosted candidates to care-agent, then combine their answers."),
    agents={"screening-agent": screening_agent,
            "scheduling-agent": scheduling_agent,
            "care-agent": care_agent},
    allowed_tools=["Agent", "Read"])              # "Agent" unlocks delegation
print("coordinator ready with:", list(COORD.agents))

---

### 🎯 Part A - decompose and delegate

**TODO 1 (about 5 minutes).** Complete `classify()`, the pure-Python query-type analysis that drives
routing. Return the set of intents present in a request, using the keyword lists provided. If nothing
matches, default to `{"screening"}`.

In [ ]:
# ===== TODO 1 - decompose by query type: detect the intents present =====
SCREEN_KW = ["screen", "resume", "cv", "qualif", "evaluate", "assess", "shortlist", "bar"]
SCHED_KW  = ["schedule", "interview", "slot", "availabil", "calendar", "reschedule", "book"]
CARE_KW   = ["upset", "frustrat", "angry", "complaint", "ghost", "rude", "unhappy", "waiting"]

def classify(request):                            # request text -> a set of intents
    text = request.lower()                        #   normalise for matching
    intents = set()
    # 👉 TODO 1a: add "screening" if any SCREEN_KW is in text
    # 👉 TODO 1b: add "scheduling" if any SCHED_KW is in text
    # 👉 TODO 1c: add "care" if any CARE_KW is in text
    return intents or {"screening"}               #   default to screening if nothing matched

for r in ["Screen candidate C1.", "Schedule an interview for C3.", "C3 is frustrated after being ghosted."]:
    print(f"{r!r:44} ->", classify(r))

**Self-check (offline).** Confirms the three intents are detected correctly.

In [ ]:
# ===== self-check for TODO 1 =====
assert classify("Screen candidate C1 against the bar.") == {"screening"}
assert classify("Schedule an interview for C3.") == {"scheduling"}
assert classify("C3 is frustrated after being ghosted.") == {"care"}
assert classify("Screen C1 and schedule their interview.") == {"screening", "scheduling"}
print("TODO 1 checks passed")

**TODO 2 (about 4 minutes).** Complete `route()`, which maps the detected intents to subagent
names so each request goes only to the subagents it needs. Use `INTENT_TO_AGENT` and return the
names sorted (so the order is stable). `FIXED_PIPELINE` is the rigid alternative it replaces.

In [ ]:
# ===== TODO 2 - route dynamically, instead of running a fixed pipeline =====
INTENT_TO_AGENT = {"screening": "screening-agent",   # map each intent to its spoke
                   "scheduling": "scheduling-agent",
                   "care": "care-agent"}
FIXED_PIPELINE = ["screening-agent", "scheduling-agent", "care-agent"]   # the rigid alternative

def route(request):                               # request -> the subagents it actually needs
    # 👉 TODO 2: return [INTENT_TO_AGENT[i] for i in sorted(classify(request))]
    return []                                     #   replace this line

for r in ["Screen candidate C1.", "Screen C1 and schedule their interview."]:
    print(f"{r!r:44}")
    print("   dynamic:", route(r), "  vs fixed:", FIXED_PIPELINE)

**Self-check (offline).**

In [ ]:
# ===== self-check for TODO 2 =====
assert route("Screen candidate C1.") == ["screening-agent"]
assert route("Screen C1 and schedule their interview.") == ["scheduling-agent", "screening-agent"]
print("TODO 2 checks passed")

**TODO 3 (about 5 minutes).** Complete `handoff()`, the self-contained packet a subagent
receives. Because a subagent inherits nothing, bundle `content` (the task in words) and `metadata`
(the candidate id and its facts), so the packet is the entire channel between coordinator and
subagent.

In [ ]:
# ===== TODO 3 - structured context passing: content + metadata =====
def handoff(request, candidate_id, intent):       # build a self-contained brief for one subagent
    facts = CANDIDATES[candidate_id]              #   the facts the subagent cannot otherwise see
    # 👉 TODO 3: return a dict with:
    #    "content":  f"{intent} task: {request}"
    #    "metadata": {"candidate_id": candidate_id, "facts": facts,
    #                 "stage_word": STAGE_NAMES[facts["stage"]], "intent": intent}
    return {}                                     #   replace this line

print(json.dumps(handoff("Screen this candidate", "C1", "screening"), indent=2))

**Self-check (offline).**

In [ ]:
# ===== self-check for TODO 3 =====
h = handoff("Screen this candidate", "C1", "screening")
assert h["metadata"]["candidate_id"] == "C1"
assert h["metadata"]["stage_word"] == "interview"          # C1 is at stage 3
assert h["metadata"]["facts"] == CANDIDATES["C1"]
assert h["content"].startswith("screening task:")
print("TODO 3 checks passed")

**This cell:** an offline **mock coordinator** (provided) that uses your `route()` to narrate
the delegation, exactly the shape the real coordinator produces. For a batch it dispatches one
subagent per candidate in a single turn and flags **parallel dispatch**.

In [ ]:
# ===== offline mock coordinator: narrates delegation using YOUR router (provided) =====
def mock_coordinate(request):                     # show which spokes a mixed request would hit
    agents = route(request)
    if len(agents) > 1:
        print(f"  parallel dispatch: {len(agents)} subagents in one turn")
    for a in agents:
        print("  -> delegate to:", a)
    return "combined answer from: " + ", ".join(agents)

def mock_triage(request):                         # fan out one triage-agent per candidate id
    ids = [c for c in re.findall(r"C\d+", request) if c in CANDIDATES]
    if len(ids) > 1:
        print(f"  PARALLEL: {len(ids)} triage-agents dispatched in one turn")
    for cid in ids:
        print("    -> triage-agent for", cid, "(", STAGE_NAMES[CANDIDATES[cid]["stage"]], ")")
    return {cid: STAGE_NAMES[CANDIDATES[cid]["stage"]] for cid in ids}

print("mixed request:")
print(mock_coordinate("Screen C1 and schedule their interview."))
print("\nbatch triage:")
print(mock_triage("Triage candidates C1, C3, and C4 independently."))

**This cell:** the **live coordinator** (needs a key and Node.js 18+). It runs the real
`query()` over the mixed request and delegates to the spokes your router picked. Offline it prints
the expected delegation instead.

In [ ]:
# ===== run the real coordinator on a mixed request =====
mixed = "Screen candidate C1, and schedule an interview for C3."
if RUN_LIVE and SDK_OK:
    async def stream_run(options, prompt):
        print("USER:", prompt); answer = ""
        async for message in query(prompt=prompt, options=options):
            if isinstance(message, AssistantMessage):
                subs = [b for b in message.content
                        if isinstance(b, ToolUseBlock) and b.name in ("Agent", "Task")]
                if len(subs) > 1: print(f"  parallel dispatch: {len(subs)} subagents in one turn")
                for b in message.content:
                    if isinstance(b, ToolUseBlock) and b.name in ("Agent", "Task"):
                        who = b.input.get("subagent_type", "?") if isinstance(b.input, dict) else "?"
                        print("  -> delegate to:", who)
                    elif isinstance(b, TextBlock): answer = b.text
            elif isinstance(message, ResultMessage): print("  (run complete)")
        print("ANSWER:", answer); return answer
    run_async(lambda: stream_run(COORD, mixed))
else:
    print("[offline] expected: delegate to screening-agent for C1 and scheduling-agent for C3,")
    print("          then a combined reply. Your route() picks:", route(mixed))

---

### 🎯 Part B - adaptive decomposition, and measure it

Two strategies plan the subtasks for a request. The **fixed pipeline** always plans the same steps;
**adaptive decomposition** plans only what the request implies (it reuses your `classify`). The
scorer decides which fits.

In [ ]:
# ===== two strategies (provided): fixed pipeline vs adaptive decomposition =====
def fixed_plan(request):                          # strategy A: always the same steps
    return {"screening", "scheduling"}            #   regardless of the ask

def adaptive_plan(request):                       # strategy B: plan only what is needed
    return classify(request)                      #   the plan IS the detected intents

print("fixed   :", fixed_plan("anything"))
print("adaptive:", adaptive_plan("Screen candidate C1."))

**This cell:** the labelled **evaluation set** (provided): recruiter requests, each with the
subtasks it truly needs. Both strategies are graded against this ground truth.

In [ ]:
# ===== the labelled requests we grade against (provided) =====
CASES = [
    ("Screen candidate C1 against the backend bar.",        {"screening"}),
    ("Schedule an interview for C3.",                       {"scheduling"}),
    ("Screen C1 and schedule their interview.",             {"screening", "scheduling"}),
    ("C3 is frustrated after being ghosted.",               {"care"}),
    ("C4 is upset we ghosted them; also assess their resume.", {"screening", "care"}),
]
print("evaluation cases:", len(CASES))

**TODO 4 (about 5 minutes).** Complete `evaluate()`. For each case, compare the strategy's plan
to the labelled truth: **spurious** steps are `plan - needed`, **missing** steps are `needed - plan`,
and a case is **exact** only when both are empty. Return the count of exact cases.

In [ ]:
# ===== TODO 4 - grade a strategy across the evaluation set =====
def evaluate(plan_fn, name):                      # plan_fn: request -> set of subtasks
    exact = 0
    for request, needed in CASES:
        plan = plan_fn(request)
        # 👉 TODO 4a: spurious = plan - needed ; missing = needed - plan
        # 👉 TODO 4b: ok = (no spurious) and (no missing) ; add ok to exact
        spurious, missing, ok = set(), set(), False   # replace these three
        print(f"  {name:8} {request[:34]!r:36} plan={sorted(plan)} "
              f"spurious={sorted(spurious)} missing={sorted(missing)}")
    print(f"  {name} quality: {exact}/{len(CASES)} exact\n")
    return exact

print("Strategy A - fixed pipeline:")
score_fixed = evaluate(fixed_plan, "fixed")
print("Strategy B - adaptive decomposition:")
score_adaptive = evaluate(adaptive_plan, "adaptive")
print("winner:", "adaptive" if score_adaptive > score_fixed else "fixed")

**Self-check (offline).** Adaptive should score a perfect 5/5 and clearly beat the fixed
pipeline.

In [ ]:
# ===== self-check for TODO 4 =====
assert evaluate(adaptive_plan, "adaptive") == 5, "adaptive should be exact on every case"
assert evaluate(fixed_plan, "fixed") < 5, "the fixed pipeline should miss several"
print("TODO 4 checks passed: adaptive beats fixed")

**Note on forked exploration.** When you are unsure which decomposition is right, fork the
coordinator's session (from Section 2) and try an alternate plan on the branch, leaving the original
intact:

```python
# given a captured session_id from a prior coordinator run:
EXPLORE = ClaudeAgentOptions(model=MODEL, resume=session_id, fork_session=True)
run_async(lambda: stream_run(EXPLORE, "Try decomposing this a different way: ..."))
```

The fork copies the history and diverges; the original session is untouched, so you can compare two
decomposition strategies as independent branches.

---

### Anti-patterns to avoid

| anti-pattern | what to do instead |
|---|---|
| one mega-agent that knows every rule | split concerns across focused subagents |
| forget `"Agent"` in `allowed_tools` | include it, or delegation is silently blocked |
| assume a subagent remembers the coordinator | pass a self-contained content + metadata packet |
| run every stage on every request | route by detected intent, not a fixed pipeline |
| triage candidates one after another | dispatch one subagent per candidate in a single turn |
| pick a strategy by gut feel | score strategies against labelled cases |
| branch by editing the live session | fork the session so the original stays intact |

**Lesson:** the coordinator is the only agent that sees the whole picture. Give each subagent a
**sharp description** (so routing is honest), a **structured hand-off** (so it has the facts), and
**isolated tools** (so concerns never bleed). Decompose by query type and route to only what is
needed; fan out independent work so subagents run in one parallel turn; and measure adaptive against
a fixed pipeline before you commit. Fixed prompt-chaining suits truly predictable tasks; adaptive
decomposition wins whenever requests vary.

---

## Recap - hub, spokes, parallelism, and evidence

| Piece | In this lab | Course topic |
|---|---|---|
| Coordinator | the hub that splits and combines | hub-and-spoke architecture (Lab 1) |
| Subagents | screening, scheduling, care via `AgentDefinition` | delegation to focused agents (Lab 1) |
| Agent tool | `allowed_tools=["Agent", ...]` | spawning subagents, older name Task (Lab 1) |
| Hand-off | `content` + `metadata` packet | explicit context passing (Lab 1) |
| Router | `classify` then `route` | decomposition by query type (Lab 1) |
| Parallel dispatch | one triage-agent per candidate in one turn | multiple Agent calls in one response (Lab 2) |
| Adaptive vs fixed | `adaptive_plan` vs `fixed_plan`, scored | comparing strategies with a rubric (Lab 2) |

**Try it next:** add a case that needs all three subtasks and watch the fixed pipeline fall further
behind. Then, live, send a frustrated, ghosted candidate message and watch care-agent get involved.